---
title: "A Year of Carsharing in Prague: Autonapůl Stats"
author: "Tomáš Trnka"
date: "2026-06-12"
categories: [data-analysis, prague, carsharing]
format:
  html:
    toc: true
    code-fold: true
    code-tools: true
execute:
  warning: false
  message: false
---

## Background

[Autonapůl](https://autonapul.zemtu.com) is a Prague-based carsharing cooperative. Members reserve
a car for a specific time slot, pick it up from its designated spot, and return it afterwards.
The fleet is spread across the city and includes everything from a small Škoda Fabia to a Renault
Trafic van.

I exported my reservation history from the API — each record covers one loan and includes the
car model, pickup location, and the exact start/end timestamps. The dataset spans roughly
**April 2023 – May 2024**, covering the full fleet, not just my own trips.

This post is a quick exploratory pass: how busy is the fleet month-to-month, which cars get
booked most, and when do people actually use carsharing?

In [ ]:
#| label: setup
#| echo: false

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

## Data loading

The raw data was fetched from the Autonapůl API as paginated JSON and then flattened to a TSV
with `jq`. The `workflow.sh` in this directory shows the exact commands. The combined file
`last_year.tsv` has one row per loan with 11 fields: reservation id, creation timestamp,
start/end timestamps, user id, vehicle id, brand, model, pickup address, city, and postcode.

In [ ]:
#| label: load-data
#| code-fold: true
#| code-summary: "Load and clean the raw TSV"

df = pd.read_csv(
    'last_year.tsv', sep='\t', header=None,
    names=['reservation_id','reservation_created','loan_start','loan_end',
           'customer_id','car_id','brand','model','car_location','car_city','postal_code']
)

# Fix UTF-8 encoding artefacts in brand names
df.loc[df['brand'].str.contains('koda', na=False) & (df['brand'] != 'Škoda'), 'brand'] = 'Škoda'

df = df.assign(
    reservation_created=pd.to_datetime(df['reservation_created'], utc=True),
    loan_start=pd.to_datetime(df['loan_start'], utc=True),
    loan_end=pd.to_datetime(df['loan_end'], utc=True),
)

df['duration_h'] = (df['loan_end'] - df['loan_start']).dt.total_seconds() / 3600
df['advance_h']  = (df['loan_start'] - df['reservation_created']).dt.total_seconds() / 3600
df['year_month'] = df['loan_start'].dt.to_period('M')
df['weekday']    = df['loan_start'].dt.day_name()
df['hour']       = df['loan_start'].dt.hour
df['car_label']  = df['brand'] + ' ' + df['model']

# Drop the partial first and last month
full_months = df.groupby('year_month').size()
full_months = full_months[full_months >= 100].index
df_full = df[df['year_month'].isin(full_months)].copy()

print(f'Total loans       : {len(df):,}')
print(f'Full months only  : {len(df_full):,}')
print(f'Unique users      : {df["customer_id"].nunique():,}')
print(f'Unique vehicles   : {df["car_id"].nunique()}')
print(f'Date range        : {df["loan_start"].min().date()} → {df["loan_end"].max().date()}')

## Monthly volume

The first thing to check is whether there is a seasonal pattern — one might expect winter
to be quieter as people avoid driving in bad weather or travel less.

In [ ]:
#| label: monthly-volume
#| fig-cap: "Number of loans per month. Partial months (Apr 2023, Jun 2024) are excluded."
#| code-fold: true

monthly = (
    df_full
    .groupby('year_month')
    .size()
    .reset_index(name='loans')
    .sort_values('year_month')
)
monthly['label'] = monthly['year_month'].dt.strftime('%b %y')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly['label'], monthly['loans'], color='steelblue', width=0.6)
ax.set_xlabel('Month')
ax.set_ylabel('Number of loans')
ax.set_title('Loans per month — Autonapůl fleet')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

There is no meaningful winter dip. Volume stays in the 320–390 band for most months,
with a noticeable spike in February 2024 (477 loans). The fleet appears to be used
year-round for practical city errands rather than seasonal leisure.

## Which cars get booked?

The fleet mixes compact city cars, mid-size hatchbacks, and a couple of specialist vehicles
(a van and an SUV). My prior was that cheaper, smaller cars would dominate.

In [ ]:
#| label: models
#| fig-cap: "Total loans by car model across the full dataset."
#| code-fold: true

model_counts = (
    df.groupby('car_label')
    .size()
    .reset_index(name='loans')
    .sort_values('loans', ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(model_counts['car_label'], model_counts['loans'], color='steelblue')
ax.set_xlabel('Number of loans')
ax.set_title('Loans by car model')
for bar, val in zip(bars, model_counts['loans']):
    ax.text(val + 10, bar.get_y() + bar.get_height() / 2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

The Škoda Fabia is far ahead with 1,493 loans (~32 % of all trips). Together the compact
segment — Fabia, Scala, and Dacia Sandero — accounts for over half of all loans. The
Renault Trafic van and Arkana SUV are niche but occupy clear use-cases (moving house,
family trips).

Part of the Fabia's lead is simply that there are more Fabias in the fleet. The
per-vehicle utilisation table below normalises for that.

In [ ]:
#| label: per-vehicle
#| tbl-cap: "Average loans per vehicle per month, sorted by utilisation."
#| code-fold: true

per_vehicle = (
    df.groupby(['car_id', 'brand', 'model', 'year_month'])
    .size()
    .reset_index(name='loans')
    .groupby(['car_id', 'brand', 'model'])
    .agg(total_loans=('loans', 'sum'), active_months=('year_month', 'nunique'))
    .assign(loans_per_month=lambda x: (x['total_loans'] / x['active_months']).round(1))
    .reset_index()
    .sort_values('loans_per_month', ascending=False)
)

per_vehicle[['brand','model','total_loans','active_months','loans_per_month']] \
    .rename(columns={
        'brand': 'Brand', 'model': 'Model',
        'total_loans': 'Total loans',
        'active_months': 'Months active',
        'loans_per_month': 'Loans / month'
    }) \
    .reset_index(drop=True)

## When do people book?

### Day of the week

In [ ]:
#| label: weekday
#| fig-cap: "Loan starts by day of the week."
#| code-fold: true

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow = (
    df.groupby('weekday')
    .size()
    .reindex(day_order)
    .reset_index(name='loans')
)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4c72b0'] * 5 + ['#dd8452'] * 2
ax.bar(dow['weekday'], dow['loans'], color=colors, width=0.6)
ax.set_xlabel('Day of week')
ax.set_ylabel('Number of loans')
ax.set_title('Loan starts by weekday')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

Wednesday and Thursday are the busiest days — not the weekend. This is the opposite of
what one might expect from a leisure-use service. Saturday is roughly on par with
Tuesday and Friday, while Sunday is the quietest day of all. The pattern suggests
that carsharing is used heavily for mid-week errands: shopping, appointments, short
trips outside the city.

### Hour of the day

In [ ]:
#| label: hour
#| fig-cap: "Loan starts by hour (local time)."
#| code-fold: true

hourly = df.groupby('hour').size().reset_index(name='loans')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hourly['hour'], hourly['loans'], color='steelblue', width=0.8)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Number of loans')
ax.set_title('Loan start time distribution')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)])
plt.tight_layout()
plt.show()

The distribution is a classic working-day bell curve peaking at 08:00. Activity drops
off steadily from mid-afternoon and is nearly zero between midnight and 05:00. Very few
night-time starts reinforces the practical-errand interpretation.

## How long are the trips?

Trip duration is the most bimodal dimension in the dataset. The median is just 4.5 hours,
but the mean is 27 hours — a classic sign of a heavy right tail.

In [ ]:
#| label: duration
#| fig-cap: "Distribution of trip durations, capped at 200 h for readability (a small number of loans exceed this)."
#| code-fold: true

durations = df['duration_h'].clip(upper=200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(durations, bins=80, color='steelblue', edgecolor='none')
axes[0].set_xlabel('Duration (hours)')
axes[0].set_ylabel('Count')
axes[0].set_title('Trip duration (capped at 200 h)')

labels = ['<1h', '1–4h', '4–8h', '8–24h', '1–3d', '3–7d', '>7d']
counts = [647, 1475, 769, 553, 522, 248, 141]
axes[1].bar(labels, counts, color='steelblue')
axes[1].set_xlabel('Duration bucket')
axes[1].set_ylabel('Count')
axes[1].set_title('Loans by duration bucket')

plt.tight_layout()
plt.show()

print(f"Median duration : {df['duration_h'].median():.1f} h")
print(f"Mean duration   : {df['duration_h'].mean():.1f} h")
print(f"90th percentile : {df['duration_h'].quantile(0.9):.1f} h")

Two usage modes are clearly visible:

- **Short errands** (under 8 hours): 64 % of all loans. These are quick trips — a supermarket
  run, a doctor's visit, picking something up across town.
- **Multi-day loans** (over 24 hours): about 23 % of loans, but they dominate total vehicle-hours.
  These are likely weekend getaways or out-of-city trips where public transport is impractical.

## How far in advance do people book?

Carsharing lives and dies by availability. If people book weeks ahead the service behaves
more like a traditional rental agency; if they book same-day it requires spare capacity at
short notice.

In [ ]:
#| label: advance-booking
#| fig-cap: "Hours between reservation creation and loan start, capped at 30 days for readability."
#| code-fold: true

advance = df['advance_h'].clip(lower=0, upper=30 * 24)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(advance, bins=60, color='#4c72b0', edgecolor='none')
ax.axvline(df['advance_h'].median(), color='crimson', linewidth=1.5,
           linestyle='--', label=f'Median ({df["advance_h"].median():.0f} h)')
ax.set_xlabel('Hours in advance')
ax.set_ylabel('Count')
ax.set_title('Advance booking time')
ax.set_xticks([0, 24, 48, 72, 96, 120, 144, 168, 336, 504, 672])
ax.set_xticklabels(['0','1d','2d','3d','4d','5d','6d','1w','2w','3w','4w'], rotation=20)
ax.legend()
plt.tight_layout()
plt.show()

print(f"Median advance  : {df['advance_h'].median():.1f} h (~{df['advance_h'].median()/24:.1f} days)")
print(f"Mean advance    : {df['advance_h'].mean():.1f} h (~{df['advance_h'].mean()/24:.1f} days)")

The median reservation is made just **12 hours** before the loan starts — roughly same-day
or the evening before. The mean is much higher (~4.5 days) because a minority of members
plan multi-day trips well in advance, pulling the average up. The Renault Trafic stands out
as the most booked-ahead vehicle (median ~12 days), which makes sense — a van is usually
needed for a specific moving day.

In [ ]:
#| label: advance-by-model
#| fig-cap: "Median advance booking time by car model."
#| code-fold: true

advance_model = (
    df[df['advance_h'] > 0]
    .groupby('car_label')['advance_h']
    .median()
    .sort_values()
    .reset_index()
)
advance_model['advance_days'] = advance_model['advance_h'] / 24

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(advance_model['car_label'], advance_model['advance_days'], color='steelblue')
ax.set_xlabel('Median advance booking (days)')
ax.set_title('How far ahead each model is booked')
plt.tight_layout()
plt.show()

## Summary

A few things stand out from this dataset:

- **No winter slowdown.** The fleet runs at roughly constant utilisation all year round.
  Carsharing in Prague is not a fair-weather habit.
- **Compact cars dominate.** The Škoda Fabia alone accounts for nearly a third of all loans.
  The cheaper and smaller the car, the more popular it tends to be.
- **Weekday-heavy.** Wednesday and Thursday see more loans than Saturday or Sunday. This
  is not primarily a weekend-leisure service.
- **Two distinct use modes.** Most trips are short (under 8 hours), but there is a clear
  second cluster of multi-day loans that dominates total vehicle-time.
- **Spontaneous bookings.** The median reservation is made only ~12 hours before pickup.
  Members treat it like an on-demand service, not a planned rental.

::: {.callout-tip collapse="true"}
## Generated with Claude Code
This analysis was developed with the help of [Claude Code](https://claude.ai/claude-code).
:::